<a href="https://colab.research.google.com/github/edaska/Stochastic_Processes_-_Optimization_in_Machine_Learning/blob/main/lab6/Lab6_RBM_DBN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1><b>Restricted Boltzmann Machine and Deep Belief Network</b></h1>



In this exercise, you will compare the accuracy of a handwritten digit classifier (dataset is available <a href="https://www.kaggle.com/datasets/avnishnish/mnist-original?resource=download">here</a>)  based on the <i>Logistic Classification</i> algorithm when it takes as input: (i) the dataset without any preprocessing by the <i>RBM</i>, (ii) the dataset after it has been processed by the <i>RBM</i>, and (iii) the dataset processed using a <i>DBN</i>, which is two stacked <i>RBMs</i>.</p>

<p align="justify">Based on the code provided, answer the following questions:</p> <ul> <li>Briefly describe how <i>RBM</i> works. What are its differences compared with a <i>Boltzmann Machine</i>?</li> <li>What is the idea behind <i>DBNs</i>, and in which cases can be used?</li> <li>Mention the main applications of <i>RBMs</i> and <i>DBNs</i>.</li><li>Compare the classification results obtained with the <i>Logistic Regression</i> algorithm without using an <i>RBM</i>, with the results obtained when an <i>RBM</i> is used, as well as when <i>RBMs</i> and <i>DBNs</i> are used for feature extraction. What do you observe regarding the accuracy of the results?</li><li>What other feature extraction or representation learning methods could be used instead of RBMs and DBNs for handwritten digit classification? Briefly compare them in terms of accuracy, computational cost, and whether they can efficiently exploit GPU acceleration.</li></ul>

In [1]:
#!/usr/bin/env python
# source: https://stackoverflow.com/questions/52166308/stacking-rbms-to-create-deep-belief-network-in-sklearn

import kagglehub
import os # Import the os module for path manipulation

# Download latest version
path = kagglehub.dataset_download("avnishnish/mnist-original")

print("Path to dataset files:", path)

import numpy as np
import matplotlib.pyplot as plt

from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import BernoulliRBM
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

def norm(arr):
    arr = arr.astype(float)
    arr -= arr.min()
    arr /= arr.max()
    return arr

if __name__ == '__main__':

    # load MNIST data set
    mnist = loadmat(os.path.join(path, "mnist-original.mat")) # Corrected path to the .mat file
    X, Y = mnist["data"].T, mnist["label"][0]

    # normalize inputs to 0-1 range
    X = norm(X)

    # split into train, validation, and test data sets
    X_train, X_test, Y_train, Y_test = train_test_split(X,       Y,       test_size=10000, random_state=0)
    X_train, X_val,  Y_train, Y_val  = train_test_split(X_train, Y_train, test_size=10000, random_state=0)

    # --------------------------------------------------------------------------------
    # set hyperparameters

    learning_rate = 0.02
    total_units   =  800
    total_epochs  =   50
    batch_size    =  128

    C = 100. # optimum for benchmark model according to sklearn docs: https://scikit-learn.org/stable/auto_examples/neural_networks/plot_rbm_logistic_classification.html#sphx-glr-auto-examples-neural-networks-plot-rbm-logistic-classification-py)

    # --------------------------------------------------------------------------------
    # construct models

    # RBM
    rbm = BernoulliRBM(n_components=total_units, learning_rate=learning_rate, batch_size=batch_size, n_iter=total_epochs, verbose=1)

    # "output layer"
    logistic = LogisticRegression(C=C, solver='lbfgs', multi_class='multinomial', max_iter=200, verbose=1)

    models = []
    models.append(Pipeline(steps=[('logistic', clone(logistic))]))                                              # base model / benchmark
    models.append(Pipeline(steps=[('rbm1', clone(rbm)), ('logistic', clone(logistic))]))                        # single RBM
    models.append(Pipeline(steps=[('rbm1', clone(rbm)), ('rbm2', clone(rbm)), ('logistic', clone(logistic))]))  # RBM stack / DBN

    # --------------------------------------------------------------------------------
    # train and evaluate models

    for model in models:
        # train
        model.fit(X_train, Y_train)

        # evaluate using validation set
        print("Model performance:\n%s\n" % (
            classification_report(Y_val, model.predict(X_val))))

Using Colab cache for faster access to the 'mnist-original' dataset.
Path to dataset files: /kaggle/input/mnist-original


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   36.8s finished


Model performance:
              precision    recall  f1-score   support

         0.0       0.95      0.96      0.96       995
         1.0       0.96      0.98      0.97      1121
         2.0       0.90      0.90      0.90      1015
         3.0       0.90      0.88      0.89      1033
         4.0       0.93      0.92      0.92       976
         5.0       0.90      0.88      0.89       884
         6.0       0.94      0.94      0.94       999
         7.0       0.92      0.93      0.92      1034
         8.0       0.88      0.86      0.87       923
         9.0       0.89      0.90      0.89      1020

    accuracy                           0.92     10000
   macro avg       0.92      0.91      0.91     10000
weighted avg       0.92      0.92      0.92     10000


[BernoulliRBM] Iteration 1, pseudo-likelihood = -139.74, time = 13.44s
[BernoulliRBM] Iteration 2, pseudo-likelihood = -120.69, time = 18.89s
[BernoulliRBM] Iteration 3, pseudo-likelihood = -106.76, time = 18.51s
[Bernoul

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   36.9s finished


Model performance:
              precision    recall  f1-score   support

         0.0       0.98      0.98      0.98       995
         1.0       0.99      0.99      0.99      1121
         2.0       0.96      0.97      0.97      1015
         3.0       0.97      0.96      0.97      1033
         4.0       0.97      0.97      0.97       976
         5.0       0.98      0.96      0.97       884
         6.0       0.98      0.97      0.98       999
         7.0       0.97      0.98      0.97      1034
         8.0       0.96      0.96      0.96       923
         9.0       0.96      0.96      0.96      1020

    accuracy                           0.97     10000
   macro avg       0.97      0.97      0.97     10000
weighted avg       0.97      0.97      0.97     10000


[BernoulliRBM] Iteration 1, pseudo-likelihood = -141.62, time = 13.45s
[BernoulliRBM] Iteration 2, pseudo-likelihood = -118.91, time = 22.11s
[BernoulliRBM] Iteration 3, pseudo-likelihood = -107.36, time = 18.77s
[Bernoul

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   33.4s finished


Model performance:
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99       995
         1.0       0.99      0.99      0.99      1121
         2.0       0.97      0.98      0.97      1015
         3.0       0.98      0.97      0.97      1033
         4.0       0.97      0.97      0.97       976
         5.0       0.97      0.98      0.97       884
         6.0       0.99      0.98      0.99       999
         7.0       0.98      0.97      0.98      1034
         8.0       0.97      0.97      0.97       923
         9.0       0.95      0.97      0.96      1020

    accuracy                           0.98     10000
   macro avg       0.98      0.98      0.98     10000
weighted avg       0.98      0.98      0.98     10000




**Briefly describe how RBM works. What are its differences compared with a Boltzmann Machine?**

An RBM (Restricted Boltzmann Machine) is a two-layer stochastic neural network with a visible layer and a hidden layer. It models data through an energy function, and the probability of a state is proportional to $e^{−E(v,h)}$
. During training, the visible units are clamped to a data sample, the hidden units are sampled from it, and then Gibbs sampling alternates between hidden and visible units to form reconstructions. The weights are updated so that the model assigns higher probability to the training data, usually with Contrastive Divergence as an approximation of maximum-likelihood learning.

The *main difference* from a Boltzmann Machine (BM) is that in an RBM there are no connections between units in the same layer. Only visible-to-hidden connections exist. Because of this restriction, hidden units are conditionally independent given the visible units, and visible units are conditionally independent given the hidden units, so inference and sampling are much faster and easier. In a general BM, same-layer connections are allowed, which makes learning much more computationally difficult.

**What is the idea behind DBNs, and in which cases can be used?**

The idea behind DBNs is to build a deep model by stacking RBMs and training them one layer at a time. Each layer learns hidden features of the previous one, so the network learns increasingly abstract representations of the data. DBNs are used for generative modeling, feature extraction, classification/regression after pretraining, non-linear dimensionality reduction, document retrieval, and sequential-data modeling.


**Mention the main applications of RBMs and DBNs.**

RBMs are used for generative modeling, feature extraction, image reconstruction/classification. DBNs are used for deep generative modeling, unsupervised pretraining for classification/regression, non-linear dimensionality reduction.

**Compare the classification results obtained with the Logistic Regression algorithm without using an RBM, with the results obtained when an RBM is used, as well as when RBMs and DBNs are used for feature extraction. What do you observe regarding the accuracy of the results?**

From the reported results, the plain Logistic Regression model gives about 92% accuracy, the model with one RBM + Logistic Regression increases it to about 97%, and the model with stacked RBMs / DBN-style feature extraction + Logistic Regression reaches about 98%.

So, the main observation is that feature extraction with RBMs improves the classification performance, and stacking RBMs improves it a bit more. In other words, the deeper feature representation helps the classifier separate the classes better. The improvement from 92% goes to 97% is large, while the extra gain from 97% goes to 98% is smaller but still positive.

**What other feature extraction or representation learning methods could be used instead of RBMs and DBNs for handwritten digit classification? Briefly compare them in terms of accuracy, computational cost, and whether they can efficiently exploit GPU acceleration.**

Instead of RBMs/DBNs, one could use PCA, autoencoders, or CNNs. PCA is the cheapest but usually less accurate because it is linear. Autoencoders learn nonlinear latent features and usually perform better, at a higher computational cost. CNNs are usually the most accurate for handwritten digits, since they exploit local image structure, but they are also more computationally demanding. Deep methods such as autoencoders and CNNs can exploit GPUs very efficiently, while PCA is usually less dependent on GPU acceleration in common implementations.

